# 03 - PHASE 0 GATE: decomposition of the efficiency gap on REAL logits

AGENTS.md Sec 5. The evidence that motivated this project came from synthetic
inject-then-recover, which is circular. Before anything else the decomposition must
replicate on REAL logits.

Measure how much of the efficiency gap is closed, SEPARATELY, by:
1. a single global **temperature** (1 free parameter),
2. a per-sample offset indexed by **free energy** `E(x) = -logsumexp(logits)` (n_bins params),
3. a per-**class** offset fit on abundant data (K params).

**PRE-REGISTERED PASS CRITERION (Sec 5):** (3) must close a gap SUBSTANTIALLY larger than
(1) and (2), with **non-overlapping CIs**. If it does not, the hypothesis that the structure
lives at the class level does not hold on real data -> **STOP, do not enter Phase 1.**

## AMENDMENT 1 (2026-08-03) - read reports/protocol_amendments.md

The first run of this notebook returned FAIL with **every** component negative
(temperature +0.014, all energy bins negative, per-class offset -5.615). A result where every
method makes sets BIGGER signals a broken measurement, not a finding.

Cause: the split-conformal quantile uses level `ceil((n+1)(1-alpha))/n`, which **depends on n**.
A 50-sample class group therefore targets that class's ~98th percentile while the pooled global
group targets the ~95th. That level mismatch - not class structure - produced the negative gaps,
and it explains the whole pattern (energy got worse as bins grew, i.e. as samples/bin shrank).

Verified on synthetic at realistic accuracy: per-class gap **-5.98 (conformal) -> +1.98
(empirical) vs +1.84 (abundant-data oracle)** - so it was the level, not finite-sample noise.

The structure measurement now uses **level-matched empirical quantiles**. Deployment still
uses the conformal quantile and its coverage validity is guarded separately by
tests/test_coverage_validity.py (Sec 8.7).

## AMENDMENT 3 (2026-08-03, APPROVED by the human)

Amendment 1 alone still gave FAIL with class_offset -0.066 / -0.097 and EVERY mechanism <= 0.
Reason: the metric was avg set size at nominal MARGINAL coverage - but marginal split-CP is
already optimal for marginal coverage, so a class-indexed mechanism CANNOT win on it. A test
that cannot return a positive is not a test.

The adopted metric is **avg set size required to reach worst-class coverage >= 1-alpha**, with
every per-group correction estimated OUT OF SAMPLE. Verified controls: structure present
+40.89, structure absent -2.52 (in-sample variants and threshold-shuffling nulls were tested
and rejected as biased). This is also the objective Sec 9 actually cares about.

**CIFAR-100 caveat:** this is the pipeline-DEBUG dataset (100 classes, 100 test images/class,
alpha=0.01 infeasible - see reports/phase0_checkpoint_gate.md). A CIFAR-100 result does NOT
decide the Phase-0 gate; Pl@ntNet does. Here we verify the code and read the direction.


## 1. Config - `# === EDIT ME ===`


In [ ]:
# === EDIT ME ===========================================================
REPO_URL   = ''
REPO_DIR   = 'foundation-cp'
DRIVE_ROOT = '/content/drive/MyDrive/pcc'
DATASET    = 'cifar100'
BACKBONE   = 'resnet50_self'

ALPHAS     = (0.05, 0.1)       # alpha=0.01 infeasible at 100 img/class (pre-registered)
N_SPLITS   = 100              # >=100 random cal/eval splits (Sec 8.4)
BIN_GRID   = (2, 5, 10, 20, 50, 100)
SEED = 42
EMB_DIR = f'{DRIVE_ROOT}/embeddings/{DATASET}/{BACKBONE}'
# =======================================================================
print('EMB_DIR =', EMB_DIR, '| alphas', ALPHAS)


## 2. Mount Drive + repo + env + seed


In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
if REPO_URL and not os.path.isdir(REPO_DIR):
    subprocess.run(['git','clone',REPO_URL,REPO_DIR], check=True)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR if os.path.isabs(REPO_DIR) else '/content/'+REPO_DIR)
os.environ['PYTHONPATH'] = os.getcwd() + os.pathsep + os.environ.get('PYTHONPATH','')
subprocess.run(['pip','install','-q','-r','requirements.txt'], check=False)
from pcc.utils.seed import set_seed; from pcc.utils.io import environment_stamp
set_seed(SEED)
print('env:', environment_stamp()['packages'])


## 3. Load TEST-set logits (the conformal cal/eval pool)


In [ ]:
import numpy as np, os
d = np.load(os.path.join(EMB_DIR,'test.npz'))
logits, labels = d['logits'], d['labels']
n, K = logits.shape
print(f'logits={logits.shape} classes={K} samples/class~{n//K}')
print('accuracy:', round(float((logits.argmax(1)==labels).mean()),4))


## 4. ADOPTED metric (Amendment 3): avg set size @ worst-class coverage >= 1-alpha

Every mechanism is held to the SAME class-conditional requirement, and every per-group
correction is estimated OUT OF SAMPLE (fit on cal, evaluated on eval). `gap_vs_global`
positive = cheaper than a global threshold. Verified controls: structure present +40.89,
structure absent -2.52; and at ~50 cal samples/class it cannot resolve (+3.56).


In [ ]:
from pcc.eval import decomposition as dc
from pcc.eval.stats import mean_ci
import numpy as np

MECHS = ['temperature'] + [f'energy_b{b}' for b in BIN_GRID] + ['class']
results = {}
for alpha in ALPHAS:
    acc = {m: [] for m in MECHS}; gsize = []
    infl = {m: [] for m in MECHS}
    rng = np.random.default_rng(SEED)
    for _ in range(N_SPLITS):
        idx = rng.permutation(n); cal, ev = idx[:n//2], idx[n//2:]
        r = dc.phase0_cc_decomposition(logits, labels, K, alpha, cal, ev,
                                       bin_grid=BIN_GRID, estimator='empirical')
        gsize.append(r['global']['avg_set_size'])
        for m in MECHS: acc[m].append(r[m]['gap_vs_global'])
        for m in MECHS: infl[m].append(r[m]['inflation'])
    results[alpha] = {m: mean_ci(v) for m, v in acc.items()}
    results[alpha]['_global_size'] = mean_ci(gsize)
    print(f'--- alpha={alpha} (global size {np.mean(gsize):.2f}) ---')
    for m in MECHS:
        v = results[alpha][m]
        print(f"  {m:16s} gap_vs_global={v['mean']:+8.3f}  "
              f"95% CI [{v['ci_low']:+.3f}, {v['ci_high']:+.3f}]  "
              f"size={np.mean(gsize)-v['mean']:7.2f}  infl={np.mean(infl[m]):.4f}")
    print('  (a mechanism whose size >> global size is being destroyed by threshold')
    print('   estimation noise: one badly-estimated class forces a large UNIFORM')
    print('   inflation that is then paid by every class. See Amendment 3 notes.)')


## 5. Gate verdict - class must dominate with non-overlapping CIs (Sec 5)


In [ ]:
verdicts = {}
for alpha in ALPHAS:
    r = results[alpha]
    cls = r['class']
    rivals = {m: r[m] for m in MECHS if m != 'class'}
    best_name = max(rivals, key=lambda k: rivals[k]['mean'])
    best = rivals[best_name]
    non_overlap = cls['ci_low'] > best['ci_high']
    verdicts[str(alpha)] = {'class_gap': cls['mean'], 'best_rival': best_name,
                            'rival_gap': best['mean'],
                            'non_overlapping_CI': bool(non_overlap),
                            'pass': bool(non_overlap and cls['mean'] > best['mean'])}
    print(f"alpha={alpha}: class={cls['mean']:+.3f} [{cls['ci_low']:+.3f},{cls['ci_high']:+.3f}] "
          f"vs best rival {best_name}={best['mean']:+.3f} "
          f"[{best['ci_low']:+.3f},{best['ci_high']:+.3f}] "
          f"-> {'PASS' if verdicts[str(alpha)]['pass'] else 'FAIL'}")

overall = 'PASS' if all(v['pass'] for v in verdicts.values()) else 'FAIL'
print('\nCIFAR-100 DEBUG direction:', overall)
print('EXPECTED to be inconclusive here: ~50 cal samples/class cannot resolve the')
print('class mechanism even when real structure exists (see Amendment 3).')


## 6. Write report


In [ ]:
import time
from pcc.utils.io import write_report
clean = {str(a): {k: {kk: float(vv) for kk, vv in v.items()}
                  for k, v in r.items()} for a, r in results.items()}
report = write_report('pcc/reports', f'03_phase0_decomposition_{DATASET}',
    hypothesis='a per-CLASS offset closes a substantially larger efficiency gap than a global '
               'temperature or a per-sample energy-indexed offset, on REAL logits',
    pass_criteria='class_offset gap > best rival AND non-overlapping 95% CIs at every alpha, '
                  'judged on the LEVEL-MATCHED (empirical) estimator per Amendment 1; energy '
                  'swept over bin counts so the win is not a parameter-count artefact. '
                  'CIFAR-100 is DEBUG ONLY and does not decide the gate.',
    config=dict(dataset=DATASET, backbone=BACKBONE, alphas=list(ALPHAS),
                n_splits=N_SPLITS, bin_grid=list(BIN_GRID),
                metric='avg_set_size_at_worst_class_coverage', estimator='empirical',
                amendments=['reports/protocol_amendments.md#amendment-1',
                            'reports/protocol_amendments.md#amendment-3']),
    seed=SEED, results={'by_alpha': clean, 'verdicts': verdicts,
                        'metric': 'avg_set_size_at_worst_class_coverage',
                        'debug_only': True},
    conclusion=f'{overall} (CIFAR-100 debug direction, empirical estimator; not the gate verdict)',
    started_at=time.time())
print('report:', report)
